# Notebook 10 — Memoria e Stato negli Agenti

**Settimana:** 10
**Blocco:** 5.2 — Agenti con persistenza
**Tempo stimato:** 4-5 ore

## Cosa costruirai

- Gestione della cronologia con finestra scorrevole e compressione tramite riassunto
- Livello di persistenza: salvare e caricare sessioni su disco
- Memoria semantica: estrarre fatti dalle conversazioni e indicizzarli in Chroma
- Un agente con memoria completa che ricorda l'utente tra sessioni
- **Sfida finale:** implementa il comando `/dimentica` che cancella tutta la memoria di un utente

## Prerequisiti

Settimane 6 e 9 completate. Chroma disponibile in `./chroma_db`.

```bash
pip install openai chromadb python-dotenv
```

---
## Cella 0 — Setup

In [ ]:
import os
import json
import uuid
import datetime
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()

cliente         = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma_cliente  = chromadb.PersistentClient(path="./chroma_db")

MODELO          = "gpt-4o-mini"
MODELO_EMBED    = "text-embedding-3-small"

# Directory di lavoro
DIR_SESIONES = Path("sesiones")
DIR_SESIONES.mkdir(exist_ok=True)

# ── Helper di base ────────────────────────────────────────────────────────────

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.3,
    max_tokens: int = 512
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(mensajes: list[dict], system: str | None = None) -> dict:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs, temperature=0.0,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

def generar_embedding(texto: str) -> list[float]:
    r = cliente.embeddings.create(model=MODELO_EMBED, input=texto)
    return r.data[0].embedding

print("Setup completato.")

---
## Cella 1 — Gestione della cronologia: finestra scorrevole e compressione

La cronologia cresce a ogni turno. Implementa due strategie per controllarla
e confronta il loro comportamento in una conversazione lunga simulata.

In [ ]:
# ── Strategia 1: finestra scorrevole ─────────────────────────────────────────

def recortar_historial(
    mensajes: list[dict],
    max_mensajes: int = 10
) -> list[dict]:
    """
    Mantiene gli ultimi max_mensajes messaggi.
    Preserva il messaggio di sistema se esiste (sempre in posizione 0).
    """
    sistema = [m for m in mensajes if m["role"] == "system"]
    resto   = [m for m in mensajes if m["role"] != "system"]

    if len(resto) <= max_mensajes:
        return mensajes

    recortados = len(resto) - max_mensajes
    return sistema + resto[-max_mensajes:]


# ── Strategia 2: compressione tramite riassunto ──────────────────────────────

SYSTEM_RESUMEN = """You are a conversation summarizer.
Create a concise summary of the conversation that preserves:
- Key facts mentioned by the user (name, preferences, goals)
- Decisions made or conclusions reached
- Topics discussed
Discard conversational filler. Maximum 3 sentences."""


def resumir_mensajes(mensajes: list[dict]) -> str:
    """Chiede all'LLM un riassunto di un insieme di messaggi."""
    texto = "\n".join(
        f"{m['role'].upper()}: {m['content'][:300]}"
        for m in mensajes
        if m["role"] != "system"
    )
    return completar(
        [{"role": "user", "content": f"Summarize this conversation:\n\n{texto}"}],
        system=SYSTEM_RESUMEN,
        temperature=0.0,
        max_tokens=200
    )


def comprimir_historial(
    mensajes: list[dict],
    umbral: int = 12,
    conservar_recientes: int = 6
) -> tuple[list[dict], bool]:
    """
    Comprime la cronologia quando supera la soglia.
    Restituisce (historial_comprimido, fue_comprimido).

    La struttura risultante è:
      [system] + [riassunto_del_passato] + [ultimi N messaggi]
    """
    sistema = [m for m in mensajes if m["role"] == "system"]
    resto   = [m for m in mensajes if m["role"] != "system"]

    if len(resto) <= umbral:
        return mensajes, False

    a_comprimir = resto[:-conservar_recientes]
    a_conservar = resto[-conservar_recientes:]

    resumen    = resumir_mensajes(a_comprimir)
    msg_resumen = {
        "role":    "system",
        "content": f"[Summary of earlier conversation]: {resumen}"
    }

    return sistema + [msg_resumen] + a_conservar, True


# ── Dimostrazione: conversazione lunga simulata ───────────────────────────────

CONVERSACION_LARGA = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Hi! My name is Raffaele and I'm learning AI Engineering."},
    {"role": "assistant", "content": "Hello Raffaele! Great topic. What are you focusing on?"},
    {"role": "user",      "content": "Right now I'm learning about RAG systems and agents."},
    {"role": "assistant", "content": "Excellent. RAG is fundamental for grounding LLMs on specific data."},
    {"role": "user",      "content": "I work at DIH DATAlife, a digital innovation hub in Galicia."},
    {"role": "assistant", "content": "Interesting! That sounds like a great environment for applied AI work."},
    {"role": "user",      "content": "We have projects in agri-food, health and forestry."},
    {"role": "assistant", "content": "Those are perfect domains for RAG: lots of domain-specific documents."},
    {"role": "user",      "content": "My goal is to have my first freelance client in 6 months."},
    {"role": "assistant", "content": "That's achievable. Focus on building a strong portfolio project."},
    {"role": "user",      "content": "What should I build for my portfolio?"},
    {"role": "assistant", "content": "A RAG chatbot over domain-specific documents is ideal for your sector."},
    {"role": "user",      "content": "I prefer Python over JavaScript for backend work."},
    {"role": "assistant", "content": "Great choice for AI engineering. FastAPI + Python is the standard stack."},
    {"role": "user",      "content": "Now, what is the capital of France?"},   # nuova domanda
]

n_original = len(CONVERSACION_LARGA)

# Applica la finestra scorrevole
recortado = recortar_historial(CONVERSACION_LARGA, max_mensajes=8)

# Applica la compressione
comprimido, fue_comprimido = comprimir_historial(CONVERSACION_LARGA, umbral=10, conservar_recientes=4)

print(f"Cronologia originale:  {n_original} messaggi")
print(f"Finestra scorrevole:   {len(recortado)} messaggi")
print(f"Compressione:          {len(comprimido)} messaggi (compresso={fue_comprimido})")

# Cosa ricorda ogni versione su Raffaele?
def preguntar_sobre_usuario(historial: list[dict], pregunta: str) -> str:
    msgs = [m for m in historial if m["role"] != "system"]  # senza system
    msgs.append({"role": "user", "content": pregunta})
    return completar(msgs, system="You are a helpful assistant.", temperature=0.0)

pregunta_memoria = "What do you know about me? Where do I work and what is my goal?"

print(f"\n{'='*60}")
print("Cosa ricorda ogni strategia sull'utente?")
print(f"{'='*60}")

r_orig  = preguntar_sobre_usuario(CONVERSACION_LARGA, pregunta_memoria)
r_recot = preguntar_sobre_usuario(recortado, pregunta_memoria)
r_comp  = preguntar_sobre_usuario(comprimido, pregunta_memoria)

print(f"\nOriginale:              {r_orig[:150]}...")
print(f"\nFinestra (ritagliata):  {r_recot[:150]}...")
print(f"\nCompressione:           {r_comp[:150]}...")

**Osservazioni** (modifica questa cella):

- La finestra scorrevole ha perso qualche dato importante?
- Il riassunto ha catturato i dati chiave (nome, azienda, obiettivo)?
- Quale strategia preferiresti per un chatbot di supporto tecnico? E per un assistente personale?

---
## Cella 2 — Persistenza delle sessioni su disco

Implementa il ciclo completo: creare sessione → salvare → caricare → continuare.

In [ ]:
@dataclass
class Sesion:
    session_id:   str
    user_id:      str
    created_at:   str
    updated_at:   str
    mensajes:     list[dict] = field(default_factory=list)
    metadatos:    dict       = field(default_factory=dict)

    @classmethod
    def nueva(cls, user_id: str, system_prompt: str | None = None) -> "Sesion":
        ahora = datetime.datetime.utcnow().isoformat()
        s     = cls(
            session_id=f"sess_{uuid.uuid4().hex[:8]}",
            user_id=user_id,
            created_at=ahora,
            updated_at=ahora,
            metadatos={"n_turnos": 0, "tokens_totales": 0, "herramientas_usadas": []}
        )
        if system_prompt:
            s.mensajes.append({"role": "system", "content": system_prompt})
        return s

    def agregar_turno(self, rol: str, contenido: str) -> None:
        self.mensajes.append({"role": rol, "content": contenido})
        self.updated_at = datetime.datetime.utcnow().isoformat()
        self.metadatos["n_turnos"] = sum(
            1 for m in self.mensajes if m["role"] == "user"
        )

    def guardar(self, directorio: Path = DIR_SESIONES) -> Path:
        ruta = directorio / f"{self.session_id}.json"
        with open(ruta, "w") as f:
            json.dump(asdict(self), f, indent=2, ensure_ascii=False)
        return ruta

    @classmethod
    def cargar(cls, session_id: str, directorio: Path = DIR_SESIONES) -> "Sesion | None":
        ruta = directorio / f"{session_id}.json"
        if not ruta.exists():
            return None
        with open(ruta) as f:
            return cls(**json.load(f))

    @classmethod
    def listar_usuario(cls, user_id: str, directorio: Path = DIR_SESIONES) -> list[dict]:
        sesiones = []
        for archivo in directorio.glob("*.json"):
            with open(archivo) as f:
                datos = json.load(f)
            if datos.get("user_id") == user_id:
                sesiones.append({
                    "session_id":  datos["session_id"],
                    "created_at":  datos["created_at"],
                    "updated_at":  datos["updated_at"],
                    "n_turnos":    datos["metadatos"].get("n_turnos", 0),
                })
        return sorted(sesiones, key=lambda x: x["updated_at"], reverse=True)


# ── Simula una sessione: crea, popola e salva ─────────────────────────────────

SYSTEM_ASISTENTE = "You are a helpful AI engineering tutor."

sesion_1 = Sesion.nueva(user_id="raffaele", system_prompt=SYSTEM_ASISTENTE)

# Simula una conversazione di 3 turni
for user_msg, asistente_resp in [
    ("What is the difference between RAG and fine-tuning?",
     completar(
         [{"role": "user", "content": "What is the difference between RAG and fine-tuning?"}],
         system=SYSTEM_ASISTENTE, max_tokens=150
     )),
    ("When should I choose RAG over fine-tuning?",
     completar(
         [{"role": "user", "content": "When should I choose RAG over fine-tuning?"}],
         system=SYSTEM_ASISTENTE, max_tokens=150
     )),
]:
    sesion_1.agregar_turno("user",      user_msg)
    sesion_1.agregar_turno("assistant", asistente_resp)

ruta_guardada = sesion_1.guardar()
print(f"Sessione salvata: {ruta_guardada}")
print(f"  Turni: {sesion_1.metadatos['n_turnos']}")
print(f"  Messaggi: {len(sesion_1.mensajes)}")

# ── Simula la ripresa della sessione ───────────────────────────────────────────
print(f"\nSimulazione di una nuova sessione (reload da disco)...")
sesion_recargada = Sesion.cargar(sesion_1.session_id)

if sesion_recargada:
    print(f"Sessione ricaricata: {sesion_recargada.session_id}")
    print(f"  Creata: {sesion_recargada.created_at}")
    print(f"  Messaggi: {len(sesion_recargada.mensajes)}")

    # Continua la conversazione da dove era rimasta
    nueva_pregunta = "Can you give me a concrete example of when RAG fails?"
    sesion_recargada.agregar_turno("user", nueva_pregunta)
    respuesta = completar(
        [m for m in sesion_recargada.mensajes if m["role"] != "system"],
        system=SYSTEM_ASISTENTE, max_tokens=150
    )
    sesion_recargada.agregar_turno("assistant", respuesta)
    sesion_recargada.guardar()

    print(f"\nDomanda di follow-up: '{nueva_pregunta}'")
    print(f"Risposta: {respuesta[:200]}...")

# ── Elenca le sessioni dell'utente ─────────────────────────────────────────────
print(f"\nSessioni dell'utente 'raffaele':")
for s in Sesion.listar_usuario("raffaele"):
    print(f"  {s['session_id']} | {s['n_turnos']} turni | aggiornata: {s['updated_at'][:16]}")

---
## Cella 3 — Memoria semantica: estrarre e recuperare fatti

La memoria semantica salva fatti rilevanti sull'utente,
indicizzati tramite embedding per recuperarli per similarità.

In [ ]:
SYSTEM_EXTRACTOR = """You are extracting durable facts about a user from a conversation.
Only extract factual, long-lasting information:
- Personal profile (name, role, company, location)
- Preferences (tools, languages, approaches they prefer)
- Goals and objectives
- Domain knowledge they have
DO NOT extract: conversational filler, temporary states, questions asked.
If there are no durable facts, return an empty list."""


@dataclass
class Hecho:
    id:         str
    user_id:    str
    texto:      str
    categoria:  str    # "perfil" | "preferencia" | "objetivo" | "conocimiento"
    session_id: str
    created_at: str


def extraer_hechos_de_sesion(sesion: Sesion) -> list[Hecho]:
    """
    Estrae fatti duraturi sull'utente da una sessione conclusa.
    """
    texto_conversacion = "\n".join(
        f"{m['role'].upper()}: {m['content'][:300]}"
        for m in sesion.mensajes
        if m["role"] in ("user", "assistant")
    )

    prompt  = f"Conversation:\n{texto_conversacion}\n\nExtract durable facts. Return JSON: {{\"hechos\": [{{\"texto\": \"...\", \"categoria\": \"perfil|preferencia|objetivo|conocimiento\"}}]}}"
    datos   = completar_json(
        [{"role": "user", "content": prompt}],
        system=SYSTEM_EXTRACTOR
    )

    ahora   = datetime.datetime.utcnow().isoformat()
    hechos  = []
    for item in datos.get("hechos", []):
        hechos.append(Hecho(
            id=f"fact_{uuid.uuid4().hex[:8]}",
            user_id=sesion.user_id,
            texto=item["texto"],
            categoria=item.get("categoria", "general"),
            session_id=sesion.session_id,
            created_at=ahora
        ))
    return hechos


def obtener_coleccion_memoria(user_id: str) -> chromadb.Collection:
    """Ottiene (o crea) la collezione di memoria semantica di un utente."""
    nombre = f"memoria_{user_id}"
    try:
        return chroma_cliente.get_collection(nombre)
    except Exception:
        return chroma_cliente.create_collection(
            nombre, metadata={"hnsw:space": "cosine"}
        )


def guardar_hechos(hechos: list[Hecho]) -> int:
    """Indicizza i fatti nella collezione di memoria semantica dell'utente."""
    if not hechos:
        return 0

    coleccion = obtener_coleccion_memoria(hechos[0].user_id)

    # Evita duplicati: genera un ID deterministico basato sul testo
    for hecho in hechos:
        hecho.id = "fact_" + hashlib.md5(hecho.texto.encode()).hexdigest()[:8]

    coleccion.upsert(
        ids=[h.id for h in hechos],
        documents=[h.texto for h in hechos],
        embeddings=[generar_embedding(h.texto) for h in hechos],
        metadatas=[{
            "user_id":    h.user_id,
            "categoria":  h.categoria,
            "session_id": h.session_id,
            "created_at": h.created_at
        } for h in hechos]
    )
    return len(hechos)


def recuperar_hechos(
    pregunta: str,
    user_id: str,
    k: int = 3,
    umbral: float = 0.60
) -> list[str]:
    """Recupera i fatti più rilevanti per la domanda attuale."""
    coleccion = obtener_coleccion_memoria(user_id)
    if coleccion.count() == 0:
        return []

    vector = generar_embedding(pregunta)
    resp   = coleccion.query(
        query_embeddings=[vector],
        n_results=min(k, coleccion.count()),
        include=["documents", "distances"],
        where={"user_id": user_id}
    )
    return [
        doc for doc, dist in zip(resp["documents"][0], resp["distances"][0])
        if (1 - dist) >= umbral
    ]


# ── Test: estrae fatti dalla sessione simulata ────────────────────────────────

# Crea una sessione con contesto ricco sull'utente
sesion_demo = Sesion.nueva(user_id="raffaele", system_prompt=SYSTEM_ASISTENTE)
for user_msg, asistente_msg in [
    ("My name is Raffaele, I work at DIH DATAlife in Galicia, Spain.",
     "Nice to meet you, Raffaele!"),
    ("I'm learning AI Engineering to become a freelance developer in 6 months.",
     "That's a great goal. Focus on building portfolio projects."),
    ("I prefer Python over JavaScript and I like FastAPI for APIs.",
     "Good choices for AI engineering work."),
    ("My main domain is agri-food and biotech.",
     "Perfect for RAG applications over scientific literature."),
]:
    sesion_demo.agregar_turno("user",      user_msg)
    sesion_demo.agregar_turno("assistant", asistente_msg)

print("Estrazione dei fatti dalla sessione...")
hechos = extraer_hechos_de_sesion(sesion_demo)
print(f"\n{len(hechos)} fatti estratti:")
for h in hechos:
    print(f"  [{h.categoria}] {h.texto}")

n_guardados = guardar_hechos(hechos)
print(f"\n{n_guardados} fatti indicizzati nella memoria semantica.")

# ── Recupero per similarità ────────────────────────────────────────────────────
print("\nTest di recupero:")
PREGUNTAS_RECUPERACION = [
    "What tools does the user prefer?",
    "What is the user's professional goal?",
    "What domain does the user work in?",
]

for p in PREGUNTAS_RECUPERACION:
    hechos_rec = recuperar_hechos(p, "raffaele", k=2)
    print(f"\n  '{p}'")
    for h in hechos_rec:
        print(f"    → {h}")

---
## Cella 4 — Agente con memoria completa

Integra i tre pezzi: cronologia con compressione + persistenza di sessione + memoria semantica.
L'agente ricorda l'utente tra sessioni e personalizza le sue risposte.

In [ ]:
class AgenteConMemoria:
    """
    Agente conversazionale con tre livelli di memoria:
    1. Cronologia della sessione attuale (con compressione automatica)
    2. Persistenza su disco (carica/salva sessioni)
    3. Memoria semantica (fatti sull'utente tra sessioni)
    """

    SYSTEM_BASE = """You are a personalized AI engineering tutor.
You remember details about the user from previous sessions.
Use the USER CONTEXT provided to personalize your responses.
Be concise and practical. Never make up facts about the user."""

    def __init__(
        self,
        user_id: str,
        session_id: str | None = None,
        max_historial: int = 12
    ):
        self.user_id       = user_id
        self.max_historial = max_historial

        # Carica una sessione esistente o ne crea una nuova
        if session_id:
            self.sesion = Sesion.cargar(session_id) or Sesion.nueva(user_id)
        else:
            self.sesion = Sesion.nueva(user_id, system_prompt=self.SYSTEM_BASE)

        print(f"Sessione: {self.sesion.session_id} | Utente: {user_id}")

    def _construir_contexto_usuario(self, pregunta: str) -> str:
        """Recupera i fatti rilevanti dalla memoria semantica per questa domanda."""
        hechos = recuperar_hechos(pregunta, self.user_id, k=4)
        if not hechos:
            return ""
        return "\nUSER CONTEXT (from memory):\n" + "\n".join(f"- {h}" for h in hechos)

    def _system_con_contexto(self, pregunta: str) -> str:
        """Costruisce il system prompt arricchito con la memoria dell'utente."""
        contexto = self._construir_contexto_usuario(pregunta)
        return self.SYSTEM_BASE + contexto

    def responder(self, pregunta: str, verbose: bool = False) -> str:
        """Genera una risposta usando la memoria completa."""

        # Aggiunge la domanda alla cronologia
        self.sesion.agregar_turno("user", pregunta)

        # Comprime se necessario
        mensajes_comprimidos, comprimido = comprimir_historial(
            self.sesion.mensajes, umbral=self.max_historial
        )
        if comprimido and verbose:
            print("  [Memoria] Cronologia compressa.")

        # System prompt arricchito con la memoria semantica
        system = self._system_con_contexto(pregunta)

        # Genera la risposta
        mensajes_sin_system = [
            m for m in mensajes_comprimidos if m["role"] != "system"
        ]
        respuesta = completar(mensajes_sin_system, system=system, max_tokens=300)

        # Aggiunge la risposta alla cronologia e salva
        self.sesion.agregar_turno("assistant", respuesta)
        if comprimido:
            self.sesion.mensajes = mensajes_comprimidos + [
                {"role": "assistant", "content": respuesta}
            ]
        self.sesion.guardar()

        return respuesta

    def cerrar_sesion(self) -> None:
        """Alla chiusura: estrae i fatti e li salva nella memoria semantica."""
        print("\nChiusura della sessione...")
        hechos = extraer_hechos_de_sesion(self.sesion)
        n      = guardar_hechos(hechos)
        print(f"  {n} nuovi fatti indicizzati nella memoria semantica.")
        self.sesion.guardar()
        print(f"  Sessione {self.sesion.session_id} salvata.")

    def historial_resumen(self) -> None:
        """Mostra la cronologia della sessione attuale."""
        turnos = [(m["role"], m["content"]) for m in self.sesion.mensajes if m["role"] != "system"]
        print(f"\nCronologia ({len(turnos)} messaggi):")
        for rol, contenido in turnos:
            print(f"  [{rol}] {contenido[:100]}..." if len(contenido) > 100 else f"  [{rol}] {contenido}")


# ── Simula due sessioni separate ───────────────────────────────────────────────

print("SESSIONE 1: Prima volta che l'utente parla con l'agente")
print("=" * 60)

agente1 = AgenteConMemoria(user_id="raffaele_test")

for pregunta in [
    "Hi! I'm Raffaele, I work in AI at DIH DATAlife in Galicia.",
    "I'm learning Python and want to become a freelance AI engineer.",
    "What topics should I focus on to get my first client quickly?",
]:
    print(f"\nTu: {pregunta}")
    r = agente1.responder(pregunta)
    print(f"Agente: {r[:200]}..." if len(r) > 200 else f"Agente: {r}")

id_sesion_1 = agente1.sesion.session_id
agente1.cerrar_sesion()

# ── Seconda sessione: l'agente dovrebbe ricordare l'utente ────────────────────
print(f"\n\nSESSIONE 2: L'utente torna (nuova sessione, stessa memoria)")
print("=" * 60)

agente2 = AgenteConMemoria(user_id="raffaele_test")

for pregunta in [
    "Hey, I'm back! Do you remember anything about me?",
    "Given my background, what project should I build first?",
]:
    print(f"\nTu: {pregunta}")
    r = agente2.responder(pregunta, verbose=True)
    print(f"Agente: {r[:300]}..." if len(r) > 300 else f"Agente: {r}")

agente2.cerrar_sesion()

---
## Sfida finale — Il comando `/dimentica`

Implementa il diritto all'oblio: quando l'utente scrive `/dimentica`,
l'agente cancella tutta la sua memoria (cronologia delle sessioni + fatti semantici).

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
class AgenteConOlvido(AgenteConMemoria):
    """
    Estende AgenteConMemoria con comandi speciali:
    /dimentica          — cancella tutta la memoria dell'utente
    /dimentica sessione — cancella solo la sessione attuale
    /dimentica fatti    — cancella solo la memoria semantica
    /cronologia         — mostra la cronologia delle sessioni
    """

    COMANDOS = ["/olvida", "/historial"]

    def _es_comando(self, texto: str) -> bool:
        return any(texto.strip().startswith(cmd) for cmd in self.COMANDOS)

    def _ejecutar_comando(self, comando: str) -> str:
        """
        Elabora un comando speciale e restituisce un messaggio di conferma.

        TODO: implementa i tre sottocomandi di /olvida:

        /olvida (senza argomenti):
          1. Cancella tutti i file .json di sessioni dell'utente in DIR_SESIONES
             (filtra solo quelli con user_id == self.user_id)
          2. Cancella la collezione di memoria semantica dell'utente in Chroma
             (nome: f"memoria_{self.user_id}")
          3. Reinizializza self.sesion con una sessione nuova
          4. Restituisce: "Memory completely erased. Starting fresh."

        /olvida sesion:
          1. Cancella solo i file di sessione dell'utente
          2. Reinizializza self.sesion
          3. Restituisce: "Session history cleared. Semantic memory preserved."

        /olvida hechos:
          1. Cancella solo la collezione Chroma dell'utente
          2. Restituisce: "Semantic memory cleared. Session history preserved."

        /historial:
          Elenca le sessioni dell'utente usando Sesion.listar_usuario()

        Suggerimento: per cancellare una collezione di Chroma:
          chroma_cliente.delete_collection(f"memoria_{self.user_id}")
        """
        partes    = comando.strip().split()
        cmd_base  = partes[0]       # "/olvida" o "/historial"
        argumento = partes[1] if len(partes) > 1 else None

        # TODO: implementa qui la logica
        raise NotImplementedError("Implementa _ejecutar_comando()")

    def responder(self, entrada: str, verbose: bool = False) -> str:
        """Rileva i comandi prima di elaborare l'input normalmente."""
        if self._es_comando(entrada):
            return self._ejecutar_comando(entrada)
        return super().responder(entrada, verbose=verbose)


# ── Test del comando /olvida ───────────────────────────────────────────────────
# Decommenta quando hai implementato _ejecutar_comando()

# print("Test del comando /olvida")
# agente_test = AgenteConOlvido(user_id="raffaele_test")

# print(agente_test.responder("My name is Raffaele."))
# print(agente_test.responder("/historial"))
# print(agente_test.responder("/olvida hechos"))
# print(agente_test.responder("/olvida"))

# # Dopo la dimenticanza completa, l'agente non dovrebbe ricordare nulla
# agente_nuevo = AgenteConOlvido(user_id="raffaele_test")
# r = agente_nuevo.responder("Do you remember anything about me?")
# print(f"Dopo /olvida: {r}")

### Estensioni opzionali

1. **Memoria selettiva:** aggiungi `/olvida <topic>` che cancella solo i fatti semantici relativi a un argomento (recupera per similarità ed elimina gli ID corrispondenti con `coleccion.delete(ids=[...])`)

2. **Esportazione della memoria:** aggiungi `/esporta` che genera un JSON con tutti i fatti conosciuti sull'utente, utile per migrare verso un'altra applicazione o per sottoporli ad audit.

3. **Più agenti con memoria condivisa:** due istanze di `AgenteConMemoria` con lo stesso `user_id` ma ruoli diversi (uno come tutor, l'altro come assistente di codice) che condividono la stessa base di fatti dell'utente in Chroma.

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 10 — Settimana 10 del Piano di Formazione AI Engineering*